# handlers

> Handler wrappers for cross-domain coordination (alignment status updates)

In [ ]:
#| default_exp combined.handlers

In [ ]:
#| export
import asyncio
from functools import wraps
from typing import Callable, Any

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.combined.step_combined import (
    render_alignment_status,
)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.routes.handlers import (
    _handle_seg_init, _handle_seg_split, _handle_seg_merge,
    _handle_seg_undo, _handle_seg_reset, _handle_seg_ai_split,
)
from cjm_fasthtml_workflow_transcript_decomp.alignment.routes.handlers import (
    _handle_align_init,
)

## Wrapper Function

Wraps mutation handlers to append alignment status OOB update.
The wrapper reads segment and VAD chunk counts from state after
the handler completes, ensuring counts reflect any mutations.

In [ ]:
#| export
def wrap_seg_mutation_handler(
    handler:Callable,  # Handler function to wrap
) -> Callable:  # Wrapped handler that appends alignment status OOB
    """Wrap a segmentation mutation handler to add alignment status OOB."""
    @wraps(handler)
    async def wrapped(workflow, request, sess, *args, **kwargs):
        # Call the original handler (async or sync)
        if asyncio.iscoroutinefunction(handler):
            result = await handler(workflow, request, sess, *args, **kwargs)
        else:
            result = handler(workflow, request, sess, *args, **kwargs)
        
        # Get counts from state for alignment status
        session_id = get_session_id(sess)
        workflow_state = workflow.state_store.get_state(
            workflow.config.workflow_id, session_id
        )
        step_states = workflow_state.get("step_states", {})
        segment_count = len(step_states.get("segmentation", {}).get("segments", []))
        chunk_count = len(step_states.get("alignment", {}).get("vad_chunks", []))
        
        # Append alignment status OOB to result
        return (*result, render_alignment_status(segment_count, chunk_count, oob=True))
    
    return wrapped

In [ ]:
#| export
def wrap_align_mutation_handler(
    handler:Callable,  # Handler function to wrap
) -> Callable:  # Wrapped handler that appends alignment status OOB
    """Wrap an alignment mutation handler to add alignment status OOB."""
    @wraps(handler)
    async def wrapped(workflow, request, sess, *args, **kwargs):
        # Call the original handler (async or sync)
        if asyncio.iscoroutinefunction(handler):
            result = await handler(workflow, request, sess, *args, **kwargs)
        else:
            result = handler(workflow, request, sess, *args, **kwargs)
        
        # Get counts from state for alignment status
        session_id = get_session_id(sess)
        workflow_state = workflow.state_store.get_state(
            workflow.config.workflow_id, session_id
        )
        step_states = workflow_state.get("step_states", {})
        segment_count = len(step_states.get("segmentation", {}).get("segments", []))
        chunk_count = len(step_states.get("alignment", {}).get("vad_chunks", []))
        
        # Append alignment status OOB to result
        return (*result, render_alignment_status(segment_count, chunk_count, oob=True))
    
    return wrapped

## Pre-Wrapped Handlers

Ready-to-use wrapped versions of mutation handlers.
These are imported by `routes/init.ipynb` for registration.

In [ ]:
#| export
# Segmentation mutation handlers (change segment count)
wrapped_seg_init = wrap_seg_mutation_handler(_handle_seg_init)
wrapped_seg_split = wrap_seg_mutation_handler(_handle_seg_split)
wrapped_seg_merge = wrap_seg_mutation_handler(_handle_seg_merge)
wrapped_seg_undo = wrap_seg_mutation_handler(_handle_seg_undo)
wrapped_seg_reset = wrap_seg_mutation_handler(_handle_seg_reset)
wrapped_seg_ai_split = wrap_seg_mutation_handler(_handle_seg_ai_split)

# Alignment mutation handlers (change chunk count)
wrapped_align_init = wrap_align_mutation_handler(_handle_align_init)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()